# Kernel Cauchy Steering

## Cell 1 — Configuration and Utilities

In [ ]:
import os, json, torch, numpy as np
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
import csv
from tqdm import tqdm
from typing import List, Dict
from sklearn.linear_model import LogisticRegression


In [ ]:
EMOTION = "SAD"  # <-- Change this only once

CONFIG = {
    "model_name": "meta-llama/Llama-3.1-8B-Instruct",
    "emotion": EMOTION,
    "target_layers": [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23],
    "top_k_layers": 6,
    "sigma_kernel": None,
    "lambda_reg": 1e-3,
    "n_epochs_dual": 300,
    "checkpoint_dir": Path("checkpoints"),
    "steering_decay": 0.85,
    "steering_min_scale": 0.3,
}
CONFIG["checkpoint_dir"].mkdir(parents=True, exist_ok=True)

def save_ckpt(data, name):
    path = CONFIG["checkpoint_dir"] / f"{name}_{CONFIG['emotion']}.pt"
    torch.save(data, path)
    print(f"✅ Saved: {path}")

def load_ckpt(name):
    path = CONFIG["checkpoint_dir"] / f"{name}_{CONFIG['emotion']}.pt"
    if path.exists():
        print(f"📂 Loaded: {path}")
        return torch.load(path, map_location="cpu", weights_only=False)
    return None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## Cell 2 — Paired Dataset Loading

In [ ]:
# 1. Build path based on emotion from CONFIG
emotion_key = CONFIG["emotion"].lower()  # "SAD" -> "sad"
dataset_path = f"{emotion_key}_dataset.json"

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}\nCheck the data/ folder and filename casing.")

print(f"📂 Loading dataset: {dataset_path}")
raw_data = load_json(dataset_path)

# 2. Normalize keys to a unified format.
pairs = []
for item in raw_data:
    if emotion_key not in item:
        raise ValueError(f"Dataset entry missing key '{emotion_key}'. Available: {list(item.keys())}")

    pairs.append({
        "prompt": item["prompt"],
        "neutral": item["neutral"],
        "emotion": item[emotion_key]
    })

print(f"✅ Loaded {len(pairs)} pairs (emotion: {CONFIG['emotion']})")

# 3. Split into train and holdout (for causal probe)
rng = np.random.RandomState(42)
idx = rng.permutation(len(pairs))
n_holdout = min(50, len(pairs) // 5)
holdout_pairs = [pairs[i] for i in idx[:n_holdout]]
train_pairs   = [pairs[i] for i in idx[n_holdout:]]

print(f"📊 Train: {len(train_pairs)}, Holdout: {len(holdout_pairs)}")


## Cell 3 — Model Loading

In [ ]:
model_name = CONFIG["model_name"]
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map=DEVICE)
model.eval()
print(f"🔹 Model loaded on {DEVICE}")


## Cell 4 — Paired Activation Extraction

In [ ]:
def extract_paired_activations(pairs: List[Dict], model, tokenizer, batch_size: int = 4):
    """
    Extracts hidden state activations for paired neutral/emotion continuations.
    Returns:
        acts: dict[layer] = Tensor (2N, hidden_dim) — mean-pooled over continuation tokens
        labels: Tensor (2N,) — 1 for emotion continuation, 0 for neutral
        layer_norms: dict[layer] = float — mean activation norm per layer (used for scaling later)
    """
    n_layers = model.config.num_hidden_layers
    layer_acts = {l: [] for l in range(n_layers)}
    layer_norm_accum = {l: [] for l in range(n_layers)}
    labels_list = []

    # Prepare samples: each pair becomes 2 samples
    samples = []
    for p in pairs:
        samples.append((p["prompt"], p["emotion"], 1))
        samples.append((p["prompt"], p["neutral"], 0))

    # Hook: capture all hidden states, compute mean later via mask
    captured = {}
    hooks = []
    for l in range(n_layers):
        def hook_fn(module, inp, out, layer_idx=l):
            h = out[0] if isinstance(out, (tuple, list)) else out
            captured[layer_idx] = h.detach()
            return out
        hooks.append(model.model.layers[l].register_forward_hook(hook_fn))

    print("🔹 Extracting paired activations (continuation-mean)...")
    try:
        for i in tqdm(range(0, len(samples), batch_size)):
            batch = samples[i:i+batch_size]
            full_texts = [p + "  " + c for p, c, _ in batch]
            prompt_lens = [len(tokenizer(p, add_special_tokens=False).input_ids) for p, _, _ in batch]

            enc = tokenizer(full_texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                model(**enc, use_cache=False)

            # Continuation token mask (excluding prompt and padding)
            attn = enc.attention_mask
            B, T = attn.shape
            cont_mask = torch.zeros_like(attn, dtype=torch.bool)
            for b in range(B):
                start = prompt_lens[b]
                end = int(attn[b].sum().item())
                if end > start:
                    cont_mask[b, start:end] = True

            for l in range(n_layers):
                h = captured[l].cpu().float()
                m = cont_mask.cpu().unsqueeze(-1).float()
                # Mean over continuation tokens
                summed = (h * m).sum(dim=1)
                count = m.sum(dim=1).clamp(min=1.0)
                mean_h = summed / count
                layer_acts[l].append(mean_h)
                # Activation norms of continuation tokens
                norms = h.norm(dim=-1)
                masked_norms = norms[cont_mask.cpu()]
                layer_norm_accum[l].append(masked_norms)

            for _, _, lab in batch:
                labels_list.append(lab)
    finally:
        for h in hooks:
            h.remove()

    acts = {l: torch.cat(layer_acts[l], dim=0) for l in range(n_layers)}
    layer_norms = {l: torch.cat(layer_norm_accum[l]).mean().item() for l in range(n_layers)}
    labels = torch.tensor(labels_list)
    return acts, labels, layer_norms

ckpt = load_ckpt("activations_paired")
if ckpt is None:
    acts, labels, layer_norms = extract_paired_activations(train_pairs, model, tokenizer)
    save_ckpt({"acts": acts, "labels": labels, "layer_norms": layer_norms}, "activations_paired")
else:
    acts, labels, layer_norms = ckpt["acts"], ckpt["labels"], ckpt["layer_norms"]

print(f"Acts shape (layer 12): {acts[12].shape}")
print(f"Layer norms: { {l: round(layer_norms[l], 2) for l in CONFIG['target_layers']} }")


## Cell 5 — Kernel Cauchy Probe & Explicit Steering Vector

In [ ]:
def train_kernel_cauchy(acts, labels, target_layers, config):
    """
    Trains a Kernel Cauchy classifier for each target layer using dual optimization.
    The Cauchy kernel is robust to outliers: K(x, x') = 1 / (1 + ||x - x'||^2 / σ^2)
    """
    dual_coefs = {}
    for l in target_layers:
        print(f"\n🔹 Layer {l}: Kernel Cauchy + Dual Logistic")
        X = acts[l].to(DEVICE).float()
        y = labels.to(DEVICE).float() * 2 - 1
        N, d = X.shape

        dists = torch.cdist(X, X)
        sigma = torch.median(dists[dists > 0]).item()
        if config["sigma_kernel"] is not None:
            sigma = config["sigma_kernel"]

        K = 1.0 / (1.0 + (dists ** 2) / (sigma ** 2))
        K = (K + K.T) / 2

        alpha = torch.zeros(N, requires_grad=True, device=DEVICE)
        opt = torch.optim.Adam([alpha], lr=1e-2)
        scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=100, gamma=0.5)

        for epoch in range(config["n_epochs_dual"]):
            opt.zero_grad()
            logits = K @ alpha
            loss = torch.log(1 + torch.exp(-y * logits)).mean()
            reg = config["lambda_reg"] * (alpha @ K @ alpha)
            (loss + reg).backward()
            opt.step()
            scheduler.step()

        X_ref = X[labels == 0].mean(dim=0)
        dual_coefs[l] = {"alpha": alpha.detach().cpu(), "sigma": sigma, "X_ref": X_ref.cpu()}
        print(f"  σ={sigma:.3f}, α-norm={alpha.norm().item():.2f}, Loss={loss.item():.4f}")
    return dual_coefs

def extract_steering_vectors(dual_coefs, acts, labels, target_layers):
    """
    Derives explicit steering vectors from the trained Kernel Cauchy model.
    The steering direction corresponds to the gradient of the kernel decision function
    with respect to the reference point, pushing neutral states toward the emotion manifold.
    """
    vectors = {}
    for l in target_layers:
        dc = dual_coefs[l]
        alpha = dc["alpha"].to(DEVICE)
        X_ref = dc["X_ref"].to(DEVICE)
        X = acts[l].to(DEVICE)
        sigma = dc["sigma"]

        diffs = X_ref.unsqueeze(0) - X
        dists_sq = (diffs ** 2).sum(dim=1)
        K_ref = 1.0 / (1.0 + dists_sq / (sigma ** 2))
        weights = -2.0 / (sigma ** 2) * alpha * (K_ref ** 2)
        v = (weights.unsqueeze(1) * diffs).sum(dim=0)

        # Sign check against a linear probe
        probe = LogisticRegression(max_iter=1000).fit(X.cpu().numpy(), labels.numpy())
        probe_w = torch.tensor(probe.coef_[0], dtype=v.dtype, device=DEVICE)
        if F.cosine_similarity(v, probe_w, dim=0) < 0:
            v = -v

        v = v / (v.norm() + 1e-8)
        vectors[l] = v.cpu()
    return vectors

ckpt = load_ckpt("dual_paired")
if ckpt is None:
    dual_coefs = train_kernel_cauchy(acts, labels, CONFIG["target_layers"], CONFIG)
    save_ckpt(dual_coefs, "dual_paired")
else:
    dual_coefs = ckpt

ckpt = load_ckpt("vectors_paired")
if ckpt is None:
    vectors = extract_steering_vectors(dual_coefs, acts, labels, CONFIG["target_layers"])
    save_ckpt(vectors, "vectors_paired")
else:
    vectors = ckpt


## Cell 6 — Causal Probe for Layer Selection

In [ ]:
def teacher_forced_logprob(model, tokenizer, prompt: str, continuation: str) -> float:
    """
    Computes the sum of log-probabilities for the continuation tokens
    when the model is teacher-forced (ground-truth tokens are fed at each step).
    This provides a stable baseline metric for causal evaluation.
    """
    full = prompt + "  " + continuation
    enc_full = tokenizer(full, return_tensors="pt").to(DEVICE)
    enc_prompt = tokenizer(prompt, return_tensors="pt")
    p_len = enc_prompt.input_ids.shape[1]
    with torch.no_grad():
        out = model(**enc_full)
    logits = out.logits[0, :-1, :]
    targets = enc_full.input_ids[0, 1:]
    logp = F.log_softmax(logits, dim=-1)
    token_logp = logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    cont_logp = token_logp[p_len-1:]
    return cont_logp.sum().item()

def causal_probe_layer(model, tokenizer, layer_idx: int, v: torch.Tensor,
                       holdout: List[Dict], eta_test: float = 2.0,
                       layer_norm: float = 1.0) -> float:
    """
    Computes ΔlogP to measure the causal effect of steering on a specific layer.
    ΔlogP = [logP(emotion|steer) - logP(neutral|steer)] - [logP(emotion|base) - logP(neutral|base)]
    A positive value indicates the steering successfully shifts probability mass toward the target emotion.
    """
    v_dev = v.to(DEVICE).to(model.dtype)
    scale = eta_test * layer_norm

    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        h = h.clone()
        h += scale * v_dev
        return (h,) + out[1:] if isinstance(out, tuple) else h

    deltas = []
    for p in holdout:
        # Baseline
        base_emotion = teacher_forced_logprob(model, tokenizer, p["prompt"], p["emotion"])
        base_neut = teacher_forced_logprob(model, tokenizer, p["prompt"], p["neutral"])
        # With steering
        h_handle = model.model.layers[layer_idx].register_forward_hook(hook)
        try:
            steer_emotion = teacher_forced_logprob(model, tokenizer, p["prompt"], p["emotion"])
            steer_neut = teacher_forced_logprob(model, tokenizer, p["prompt"], p["neutral"])
        finally:
            h_handle.remove()
        # ΔlogP favoring emotion
        deltas.append((steer_emotion - steer_neut) - (base_emotion - base_neut))
    return float(np.mean(deltas))

ckpt = load_ckpt("causal_scores")
if ckpt is None:
    causal_scores = {}
    for l in CONFIG["target_layers"]:
        score = causal_probe_layer(model, tokenizer, l, vectors[l], holdout_pairs,
                                    eta_test=2.0, layer_norm=layer_norms[l])
        causal_scores[l] = score
        print(f"  Layer {l}: ΔlogP(emotion-neutral) = {score:+.4f}")
    save_ckpt(causal_scores, "causal_scores")
else:
    causal_scores = ckpt
    for l, s in causal_scores.items():
        print(f"  Layer {l}: ΔlogP = {s:+.4f}")

# Select top-K layers with positive effect
sorted_layers = sorted(causal_scores.items(), key=lambda kv: kv[1], reverse=True)
selected_layers = [l for l, s in sorted_layers if s > 0][:CONFIG["top_k_layers"]]
if not selected_layers:
    print("⚠️ No layer gave a positive causal effect — fallback to top-K by absolute value")
    selected_layers = [l for l, _ in sorted_layers[:CONFIG["top_k_layers"]]]
selected_layers = sorted(selected_layers)
print(f"\n🎯 Selected layers (by causal probe): {selected_layers}")
CONFIG["selected_layers"] = selected_layers


## Cell 7 — Inference with Improved Steering

In [ ]:
def apply_steering_hooks(model, layers, vectors, eta_base, layer_norms, config):
    """
    Registers forward hooks to inject steering vectors during text generation.
    Implements a decay schedule over generation steps to prevent over-steering
    and maintain coherence in longer outputs.
    """
    hooks = []
    decay = config["steering_decay"]
    min_scale = config["steering_min_scale"]
    # Hook call counter per layer to track decoding step
    call_counters = {l: [0] for l in layers}

    for l in layers:
        v = vectors[l].to(model.device).to(model.dtype)
        # FIX 2: scale in units of this layer's activation norm
        base_scale = eta_base * layer_norms[l]

        def make_hook(vec, layer_id, base):
            def hook(module, inp, out):
                h = out[0] if isinstance(out, tuple) else out
                h = h.clone()
                T = h.shape[1]
                step = call_counters[layer_id][0]
                # FIX 5: decay along sequence position
                if T == 1:
                    pos_scale = max(decay ** step, min_scale)
                    h[:, 0, :] += base * pos_scale * vec
                    call_counters[layer_id][0] += 1
                else:
                    # prefill: apply uniformly to all tokens
                    for t in range(T):
                        pos_scale = max(decay ** t, min_scale)
                        h[:, t, :] += base * pos_scale * vec
                    call_counters[layer_id][0] = T
                return (h,) + out[1:] if isinstance(out, tuple) else h
            return hook

        hooks.append(model.model.layers[l].register_forward_hook(make_hook(v, l, base_scale)))
    return hooks

def generate_response(prompt, model, tokenizer, max_new=80, temp=0.7):
    """Standard autoregressive generation with temperature sampling."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=True,
                             temperature=temp, pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip() if text.startswith(prompt) else text


In [ ]:
TEST_PROMPTS = [
    "Tell a short story about a person's choice.",
    "Write a movie review in 2-3 sentences.",
    "What do you think about the importance of moral principles?",
    "What would you do if you had one free day with no obligations?",
    "If you could instantly learn one skill without effort, what would it be and why?",
    "Invent a new holiday and describe how people celebrate it."
]

N_REPEATS = 1
RESULT_CSV = "results/generation_results.csv"
os.makedirs("results", exist_ok=True)

# Explicit schedule: (eta_value, num_repeats)
ETA_SCHEDULE = [
    (0.0, 1), (0.1, 1), (0.2, 1), (0.25, 1),
    (0.3, N_REPEATS), (0.35, 1), (0.4, N_REPEATS), (0.45, N_REPEATS),
    (0.5, N_REPEATS), (0.55, N_REPEATS), (0.6, N_REPEATS),
    (0.7, 1), (0.8, 1), (0.9, 1)
]

with open(RESULT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['prompt_id', 'prompt', 'eta_base', 'repeat', 'response', 'length', 'rep_ratio'])

    for p_idx, prompt in enumerate(TEST_PROMPTS, 1):
        print(f"\n📝 Prompt {p_idx}: '{prompt[:60]}...'")

        # Iterate through the schedule
        for eta, current_repeats in tqdm(ETA_SCHEDULE, desc="Eta sweep", leave=False):
            last_resp = ""
            for rep in range(current_repeats):
                hooks = apply_steering_hooks(model, selected_layers, vectors, eta, layer_norms, CONFIG)
                try:
                    resp = generate_response(prompt, model, tokenizer)
                    last_resp = resp

                    words = resp.split()
                    rep_ratio = 1.0 - (len(set(words)) / len(words)) if words else 0.0
                    writer.writerow([p_idx, prompt, eta, rep, resp, len(words), round(rep_ratio, 3)])
                    f.flush()
                finally:
                    # Ensure hooks are removed even on generation error
                    for h in hooks:
                        h.remove()

            # Print the last repeat's result for the current eta
            print(f"  🔹 η_base={eta} (×{current_repeats}): {last_resp[:90]}...")

print(f"\n✅ Sweep complete: {RESULT_CSV}")
